In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M25.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.769833608749828, 'n_it': 0.39682044359765484}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 700

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[15.705382836380675, 13.321558607066374, 16.292797988981683, 13.894219360996543, 13.405956978617592, 14.457243187891853, 13.777792457545065, 14.327802668209365, 13.864538848921608, 13.457820116545967, 13.78832806933362, 14.481095848535896, 14.403483664064161, 13.714937123078773, 17.292061791820245, 14.673119988437119, 13.877220784828538, 14.268296362347625, 15.2829620532183, 13.67110220612929, 13.606851026863753, 13.804773633745725, 14.61288393074034, 13.720224773071676, 14.845666834591238, 13.774006219761, 16.242175116876012, 15.026715735845503, 15.128804336252701, 13.711089394710571, 13.988748962760658, 13.466449533083221, 15.141428147706073, 17.163045387293376, 13.27240014197535, 18.20404843302559, 14.165390289100092, 13.819038893115996, 16.98826328804215, 13.581795146850395, 14.262708417403106, 13.563869927934117, 14.645169896120445, 15.230527903078546, 13.890499726097449, 13.573295589453604, 13.721887955282384, 13.864573338255681, 15.970678632966523, 13.569203816441819, 15.5153192

In [5]:
np.average(y_max_arr)

np.float64(14.974936026407088)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)